# Scene Graph Generation & Failure Summarization Pipeline

**Tim**

This notebook runs the full REFLECT pipeline end-to-end: from raw RGB-D video frames to hierarchical state summaries and LLM-based failure reasoning. The pipeline uses MDETR for object detection, CLIP for state classification, Open3D for 3D scene graph construction, and GPT-4 for failure analysis. A toggle at the top controls whether audio events (via AudioCLIP) are included in the summaries.

### Dataset & Background

The pipeline operates on the **RoboFail** dataset — robot manipulation recordings stored as zarr archives containing RGB-D video, gripper state, and action stage annotations. Each task has a JSON definition specifying the task name, object list, success condition, ground-truth failure reason, and action sequence.

The goal is to detect *where* and *why* a robot task failed by building a hierarchical summary:
- **Level 0 (L0):** Dense per-frame local scene graphs with spatial relations
- **Level 1 (L1):** Key-frame summaries combining action labels, visual scene text, and optionally audio events
- **Level 2 (L2):** Subgoal-level summaries filtered to interaction endpoints

These summaries feed into GPT-4 prompts for failure identification and reasoning.

#### References
- [REFLECT paper / RoboFail dataset](https://github.com/real-stanford/reflect)
- [MDETR: Modulated Detection for End-to-End Multi-Modal Understanding](https://arxiv.org/abs/2104.12763)
- [AudioCLIP](https://github.com/AndreyGuzhov/AudioCLIP)

In [ ]:
# Standard library
import os
import json
import pickle
import random
from argparse import Namespace

# Numerical / vision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import open3d as o3d
import zarr
from imagecodecs import imread

# Pipeline modules
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from mdetr_object_detector import plot_inference_segmentation, seg_model
from real_world_scene_graph import Node as SceneGraphNode, SceneGraph
from real_world_get_local_sg import get_scene_graph
from real_world_hierarchical_prompt import (
    get_scene_text, run_reasoning, run_wo_sound,
    get_interact_actions, read_zarr, create_folders, config_parser
)
from real_world_utils import get_robot_plan
from LLM.prompt import LLMPrompter
from main.utils import convert_step_to_timestep

# Reproducibility
np.random.seed(91)
random.seed(91)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

### Configuration

All pipeline parameters live in one place. The most important toggle is `USE_AUDIO` — set it to `False` to strip auditory observations from the summaries and run the ablation without sound. `TASK_INDICES` controls which tasks from the task JSON to process.

In [ ]:
# ──────────────────────────────────────────────
# Main toggles
# ──────────────────────────────────────────────
USE_AUDIO       = True          # False → run without audio (ablation_type=0, audio_ver=0)
TASK_INDICES    = [1]           # list of task numbers to process, or [0] for all (1-30)
OBJ_DETECTOR    = "mdetr"       # "mdetr" or "detic"
ABLATION_TYPE   = 0             # 0=full, 3=L2-only, 5=BLIP2

# Build args namespace to match the original argparse interface
args = Namespace(
    tasks=list(map(str, TASK_INDICES)),
    folder_name="",
    obj_det=OBJ_DETECTOR,
    audio_ver=1 if USE_AUDIO else 0,
    ablation_type=ABLATION_TYPE,
)

print(f"Audio: {'ON' if USE_AUDIO else 'OFF'}")
print(f"Tasks: {TASK_INDICES}")
print(f"Ablation type: {ABLATION_TYPE}")

### Load Task Definitions

The task JSON contains per-task metadata: object lists, success conditions, action sequences, and ground-truth failure annotations. The task list is resolved here so we can iterate through it in the following steps.

In [ ]:
with open("real_world/tasks_real_world.json", "r") as f:
    tasks_json = json.load(f)

task_list = list(map(int, args.tasks))
if task_list == [0]:
    task_list = list(range(1, 31))

print(f"Processing {len(task_list)} task(s): {task_list}")
print(f"First task keys: {list(tasks_json['Task ' + str(task_list[0])].keys())}")

### Initialize LLM Prompter

The `LLMPrompter` wraps the OpenAI GPT-4 API and handles prompt formatting, querying, and optional response caching. It is used in the final reasoning stage to identify failure reasons and failure steps.

In [ ]:
llm_prompter = LLMPrompter(gpt_version="gpt-4")
print("LLM prompter ready.")

## Pipeline Loop

The main loop processes one task at a time through four stages:
1. **Setup** — read zarr metadata, extract interaction actions and (optionally) audio events
2. **Level-0: Dense scene graphs** — run MDETR + CLIP on every interaction frame, build local + global scene graphs
3. **Level-1 & Level-2 summaries** — select key frames and generate text summaries at two abstraction levels
4. **LLM reasoning** — prompt GPT-4 to identify and explain the failure

If `USE_AUDIO` is `False`, the pipeline calls `run_wo_sound` which strips auditory observations from existing summaries before reasoning.

In [ ]:
for task_idx in task_list:
    task_info = tasks_json[f"Task {task_idx}"]
    print(f"\n{'='*60}")
    print(f"TASK {task_idx}: {task_info.get('name', 'unknown')}")
    print(f"Objects: {task_info.get('object_list', [])}")
    print(f"Success condition: {task_info.get('success_condition', 'N/A')}")
    print(f"{'='*60}")

    if args.audio_ver == 0:
        # Without audio — strips sound from existing summaries, then runs reasoning
        run_wo_sound(args, task_info)
    else:
        # Full pipeline with audio
        from real_world_hierarchical_prompt import run_real_world_pipeline
        run_real_world_pipeline(args, task_info)

    print(f"\nTask {task_idx} complete.")

### Inspect Results

After the pipeline finishes, each task folder contains the generated summaries and reasoning outputs. This cell loads and displays them so you can quickly verify the results without digging through files.

In [ ]:
for task_idx in task_list:
    task_info = tasks_json[f"Task {task_idx}"]
    folder = task_info["general_folder_name"]
    base = f"real_world/state_summary/{folder}"

    print(f"\n{'='*60}")
    print(f"RESULTS — Task {task_idx} ({folder})")
    print(f"{'='*60}")

    # Show L2 summary
    l2_suffix = "_wo_sound" if not USE_AUDIO else ""
    l2_path = f"{base}/state_summary_L2{l2_suffix}.txt"
    if os.path.exists(l2_path):
        with open(l2_path) as f:
            print("\n--- L2 Summary (subgoal level) ---")
            print(f.read())

    # Show L1 summary
    l1_path = f"{base}/state_summary_L1{l2_suffix}.txt"
    if os.path.exists(l1_path):
        with open(l1_path) as f:
            print("\n--- L1 Summary (key-frame level) ---")
            print(f.read())

    # Show reasoning output
    if not USE_AUDIO:
        reasoning_name = "reasoning-wo-sound.json"
    else:
        reasoning_name = "reasoning.json"
    reason_path = f"{base}/{reasoning_name}"
    if os.path.exists(reason_path):
        with open(reason_path) as f:
            reasoning = json.load(f)
        print(f"\n--- Failure Reasoning ---")
        print(f"Predicted reason : {reasoning.get('pred_failure_reason', 'N/A')}")
        print(f"Predicted step   : {reasoning.get('pred_failure_step', 'N/A')}")
        print(f"Ground-truth     : {reasoning.get('gt_failure_reason', 'N/A')}")
        print(f"GT step          : {reasoning.get('gt_failure_step', 'N/A')}")

### Visualize a Local Scene Graph

To get a sense of what the scene graph captures, this cell loads a single local graph from disk and prints both the detected nodes (objects + their states) and the spatial edges between them. If a corresponding RGB frame exists, it is displayed alongside.

In [ ]:
# Pick the first task and its first key frame for visualization
task_info = tasks_json[f"Task {task_list[0]}"]
folder = task_info["general_folder_name"]
sg_dir = f"real_world/state_summary/{folder}/local_graphs"

if os.path.exists(sg_dir):
    sg_files = sorted(os.listdir(sg_dir), key=lambda x: int(x.split("_")[-1].split(".")[0]))
    if sg_files:
        sg_path = os.path.join(sg_dir, sg_files[-1])
        step = sg_files[-1].split("_")[-1].split(".")[0]
        with open(sg_path, "rb") as f:
            local_sg = pickle.load(f)

        print(f"Local scene graph at step {step}:")
        print(local_sg)

        # Try to show the corresponding RGB frame
        img_path = f"real_world/images/{folder}/rgb_{step}.png"
        if os.path.exists(img_path):
            fig, ax = plt.subplots(1, 1, figsize=(10, 6))
            ax.imshow(Image.open(img_path))
            ax.set_title(f"Frame {step}")
            ax.axis("off")
            plt.tight_layout()
            plt.show()
else:
    print(f"No local graphs found at {sg_dir} — run the pipeline first.")

### Visualize the Global Scene Graph

The global scene graph aggregates point clouds across all frames and captures the overall spatial layout of the workspace. Comparing it to a local graph shows how the global view smooths over per-frame noise.

In [ ]:
global_sg_path = f"real_world/state_summary/{folder}/global_sg.pkl"
if os.path.exists(global_sg_path):
    with open(global_sg_path, "rb") as f:
        global_sg = pickle.load(f)
    print("Global scene graph:")
    print(global_sg)
else:
    print("Global scene graph not found — run the pipeline first.")

### Run a Single-Frame Object Detection (Optional)

If you want to inspect the MDETR detections on a single frame without running the full pipeline, this cell loads one image, runs segmentation for each object in the task's object list, and shows the annotated result. Useful for debugging detection quality before committing to a full run.

In [ ]:
# Set to True to run this optional cell
RUN_SINGLE_FRAME_DET = False

if RUN_SINGLE_FRAME_DET:
    task_info = tasks_json[f"Task {task_list[0]}"]
    folder = task_info["general_folder_name"]
    object_list = task_info["object_list"]

    # Pick a frame index (adjust as needed)
    frame_idx = 0
    rgb = imread(f"real_world/data/{folder}/videos/0/0/color/{frame_idx}.0.0.0")
    im = Image.fromarray(rgb)

    print(f"Running MDETR on frame {frame_idx} for objects: {object_list}")
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))

    for obj_name in object_list:
        retval = plot_inference_segmentation(im, obj_name, seg_model)
        im = retval["im"]

    ax.imshow(im)
    ax.set_title(f"MDETR detections — frame {frame_idx}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Single-frame detection skipped. Set RUN_SINGLE_FRAME_DET = True to enable.")

## Conclusion and Reflection

This notebook wires together the full REFLECT pipeline: MDETR object detection → CLIP-based state classification → 3D scene graph construction → hierarchical summarization → GPT-4 failure reasoning. The `USE_AUDIO` toggle makes it straightforward to compare the with-audio and without-audio ablations without changing any pipeline code.

The most fragile part of the pipeline is the object detection confidence thresholds — the 0.96 probability cutoff in MDETR and the 0.23 CLIP confirmation threshold are tuned to specific tasks (the code itself has comments about adjusting for `sauteeCarrot4`). Any new task or camera setup would likely need threshold re-tuning.

Running through the code, it becomes clear how much the pipeline relies on caching: almost every stage checks for existing pickle/txt files before recomputing. This is practical for iterating on later stages without re-running detection, but it also means stale cache files can silently produce wrong results if earlier stages change. A `--force-recompute` flag or explicit cache invalidation would make this safer to work with.